In [1]:
!git clone https://github.com/Raocp/PINN-laminar-flow.git
!pip install numpy==1.26.4 pyDOE==0.3.8
!find ./PINN-laminar-flow -type f -name "*.py" -exec sed -i 's/import tensorflow as tf/import tensorflow.compat.v1 as tf\ntf.disable_v2_behavior()/g' {} +

Cloning into 'PINN-laminar-flow'...
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 41 (delta 14), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (41/41), 4.70 MiB | 12.62 MiB/s, done.
Resolving deltas: 100% (14/14), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 81.3 MB/s eta 0:00:00:00:0100:01
  Created wheel for pyDOE: filename=pyDOE-0.3.8-py3-none-any.whl size=18170 sha256=7828a42bf5633b630b834634215d1191c7eedb05cf4508688834caabf4c7e3f4
  Stored in directory: /root/.cache/pip/wheels/96/b9/5d/1138ea8c8f212bce6e97ae58847b7cc323145b3277f2129e2b
Successfully built pyDOE
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's depen

In [2]:
import os

patch = """
# --- TF.CONTRIB POLYFILL ---
class DummyOpt:
    def __init__(self, *args, **kwargs): pass
    def minimize(self, *args, **kwargs):
        print("Skipping L-BFGS optimization (tf.contrib deprecated).")
class OptDummy:
    ScipyOptimizerInterface = DummyOpt
class ContribDummy:
    opt = OptDummy
tf.contrib = ContribDummy
# ---------------------------
"""

# Walk through the repository (Updated for Kaggle's file path)
for root, dirs, files in os.walk("/kaggle/working/PINN-laminar-flow"):
    for file in files:
        if file.endswith(".py"):
            filepath = os.path.join(root, file)
            with open(filepath, "r") as f:
                content = f.read()

            if "tf.contrib" in content and "ContribDummy" not in content:
                content = content.replace("tf.disable_v2_behavior()", "tf.disable_v2_behavior()\n" + patch)
                with open(filepath, "w") as f:
                    f.write(content)

print("tf.contrib polyfill applied successfully!")

tf.contrib polyfill applied successfully!


In [3]:
# Run Steady Model
%cd /kaggle/working/PINN-laminar-flow/PINN_steady
!python SteadyFlowCylinder_mixed.py

/kaggle/working/PINN-laminar-flow/PINN_steady
/kaggle/working/PINN-laminar-flow/PINN_steady/SteadyFlowCylinder_mixed.py:435: SyntaxWarning: invalid escape sequence '\p'
  def multiple_formatter(denominator=2, number=np.pi, latex='\pi'):
/kaggle/working/PINN-laminar-flow/PINN_steady/SteadyFlowCylinder_mixed.py:464: SyntaxWarning: invalid escape sequence '\p'
  def __init__(self, denominator=2, number=np.pi, latex='\pi'):
2026-05-20 17:16:44.202616: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779297404.618164     122 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779297404.740473     122 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1

In [4]:
# Run Unsteady Model 
%cd /kaggle/working/PINN-laminar-flow/PINN_unsteady
# (Make sure to verify this exact filename in the repo folder!)
!python TransientFlowCylinder.py

/kaggle/working/PINN-laminar-flow/PINN_unsteady
2026-05-20 17:47:04.499010: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779299224.523317     179 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779299224.531178     179 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779299224.550004     179 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779299224.550057     179 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779299224.550064     179 c